In [ ]:
// ==== Setup ====

import java.util.Arrays;

interface MyListDouble {
    // appends e at the end
    void add(double e);

    // inserts e at pos, elements pos..size-1 move up by one
    // valid: 0 <= pos <= size
    void add(int pos, double e);

    // removes and returns the element at pos, elements pos+1..size-1 move down by one
    // valid: 0 <= pos < size
    double remove(int pos);

    // valid: 0 <= pos < size
    double get(int pos);

    // replaces the element at pos, returns the old one
    // valid: 0 <= pos < size
    double set(int pos, double e);

    // position of the first occurrence of e, or -1
    int indexOf(double e);

    int size();
    boolean isEmpty();
    void clear();
}

## Cost so far

| operation | cost |
|---|---|
| `get`, `set`, `size`, `isEmpty` | `O(1)` |
| `indexOf` | `O(N)` |
| `add(e)` | `O(N)` worst case, `O(1)` amortized |
| `clear` | `O(1)` |

# Array-based list II

## Inserting at a position

- Valid: `0 <= pos <= size` (`pos == size` is an append)
- If full, grow first
- Move `pos..size-1` one position up, then write `data[pos]`

Should we shift left to right, or right to left?

```java
public void add(int pos, double e) {
    if (pos < 0 || pos > size)                  // pos == size is allowed
        throw new IndexOutOfBoundsException("pos " + pos + ", size " + size);
    if (size == data.length) grow();
    for (int i = size; i > pos; i--)
        data[i] = data[i - 1];
    data[pos] = e;
    size++;
}
```

## Removing at a position

- Valid: `0 <= pos < size`
- Save the element, move `pos+1..size-1` one position down

```java
public double remove(int pos) {
    checkIndex(pos);
    double removed = data[pos];
    for (int i = pos; i < size - 1; i++)
        data[i] = data[i + 1];
    size--;
    return removed;
}
```

### With `System.arraycopy`

```java
System.arraycopy(data, pos, data, pos + 1, size - pos);       // add: shift up
System.arraycopy(data, pos + 1, data, pos, size - pos - 1);   // remove: shift down
```

- Source and destination overlap: handled correctly, as if copied through a temporary array.
- Same `O(N)`, faster in practice.

## The complete class

In [ ]:
class MyArrayListDouble implements MyListDouble {
    private double[] data;
    private int size;

    MyArrayListDouble(int capacity) {
        if (capacity < 1) throw new IllegalArgumentException("capacity must be positive");
        data = new double[capacity];
        size = 0;
    }

    MyArrayListDouble() { this(10); }

    public int size() { return size; }
    public boolean isEmpty() { return size == 0; }
    int capacity() { return data.length; }       // for our demos; java.util.ArrayList does not expose it

    private void checkIndex(int pos) {
        if (pos < 0 || pos >= size)
            throw new IndexOutOfBoundsException("pos " + pos + ", size " + size);
    }

    public double get(int pos) {
        checkIndex(pos);
        return data[pos];
    }

    public double set(int pos, double e) {
        checkIndex(pos);
        double old = data[pos];
        data[pos] = e;
        return old;
    }

    public int indexOf(double e) {
        for (int i = 0; i < size; i++)
            if (data[i] == e) return i;
        return -1;
    }

    private void grow() {
        data = Arrays.copyOf(data, 2 * data.length);
    }

    public void add(double e) {
        if (size == data.length) grow();
        data[size] = e;
        size++;
    }
    public void add(int pos, double e) {             // <===
        if (pos < 0 || pos > size)                  // pos == size is allowed
            throw new IndexOutOfBoundsException("pos " + pos + ", size " + size);
        if (size == data.length) grow();
        for (int i = size; i > pos; i--)
            data[i] = data[i - 1];
        data[pos] = e;
        size++;
    }
    public double remove(int pos) {                   // <===
        checkIndex(pos);
        double removed = data[pos];
        for (int i = pos; i < size - 1; i++)
            data[i] = data[i + 1];
        size--;
        return removed;
    }

    public void clear() {
        size = 0;
    }

    @Override
    public String toString() {
        return Arrays.toString(Arrays.copyOf(data, size));
    }
}

In [ ]:
MyArrayListDouble nums = new MyArrayListDouble(4);
for (int i = 1; i <= 5; i++) nums.add(i * 10);
System.out.println(nums);

nums.add(0, 5);                 // front
nums.add(3, 25);                // middle
nums.add(nums.size(), 99);      // end
System.out.println(nums);

System.out.println(nums.remove(0) + " " + nums.remove(2) + " " + nums);

## Cost

`add(pos, e)` moves `size - pos` elements, `remove(pos)` moves `size - pos - 1`.

| | front | middle | back |
|---|---|---|---|
| `add` | `O(N)` | `O(N)` | `O(1)` amortized |
| `remove` | `O(N)` | `O(N)` | `O(1)` |

`N` inserts at the front vs. at the back (wall-clock time, noisy: only the trend matters).

In [ ]:
long timeAdds(int n, boolean front) {
    MyArrayListDouble l = new MyArrayListDouble();
    long start = System.nanoTime();
    for (int i = 0; i < n; i++) {
        if (front) l.add(0, i);
        else       l.add(i);
    }
    return (System.nanoTime() - start) / 1_000_000;    // milliseconds
}

System.out.printf("%8s %10s %10s%n", "N", "front ms", "back ms");
for (int n = 20_000; n <= 80_000; n *= 2)
    System.out.printf("%,8d %10d %10d%n", n, timeAdds(n, true), timeAdds(n, false));

A program that mostly inserts at the front: how would you change the array-based list?

# `java.util.ArrayList`

## The `List` interface

- `java.util.List<E>`: the ADT. `ArrayList<E>` and `LinkedList<E>` implement it.
- Declare variables by the interface: `List<String> names = new ArrayList<>();`

Why `List` on the left and not `ArrayList`?

| ours | `java.util.List` |
|---|---|
| `void add(double e)` | `boolean add(E e)` |
| `void add(int pos, double e)` | `void add(int index, E e)` |
| `double remove(int pos)` | `E remove(int index)` |
| | `boolean remove(Object o)` |
| `get`, `set`, `size`, `isEmpty`, `clear` | same |
| `int indexOf(double e)` | `int indexOf(Object o)` |
| | `contains`, `sort`, `equals`, `iterator`, ... |

## Inside `ArrayList`

From the JDK source:
- data field: `Object[] elementData`
- `new ArrayList<>()`: empty array at first, capacity 10 on the first `add`
- growth: `newCapacity = oldCapacity + (oldCapacity >> 1)`, i.e. x1.5
- shifting: `System.arraycopy`
- `new ArrayList<>(n)`: initial capacity `n`
- `ensureCapacity(n)`: grow ahead of a known number of appends
- `trimToSize()`: shrink capacity down to size; never done automatically

**Capacity**: length of the array. **Size**: number of elements. `capacity >= size`.

## Equality

`List.equals`: **same size**, and elements pairwise equal **in the same order**.
Any two `List`s, whatever the implementation.

In [ ]:
import java.util.List;
import java.util.ArrayList;
import java.util.LinkedList;

List<Integer> x = new ArrayList<>(List.of(1, 2, 3));
List<Integer> y = new LinkedList<>(List.of(1, 2, 3));
List<Integer> z = List.of(3, 2, 1);
System.out.println(x.equals(y) + " " + x.equals(z) + " " + (x == y));

The same check for our lists, with `get(i)`:

In [ ]:
boolean listEquals(MyListDouble a, MyListDouble b) {
    if (a.size() != b.size()) return false;
    for (int i = 0; i < a.size(); i++)
        if (a.get(i) != b.get(i)) return false;
    return true;
}

MyArrayListDouble p = new MyArrayListDouble(), q = new MyArrayListDouble();
for (int i = 1; i <= 3; i++) { p.add(i); q.add(i); }
System.out.println(listEquals(p, q));
q.set(2, 7);
System.out.println(listEquals(p, q));

- Cost for two array lists?
- Cost for two linked lists, where `get(i)` walks the list from one end?

## Summary

| operation | `ArrayList` |
|---|---|
| `get(i)`, `set(i, e)` | `O(1)` |
| `add(e)` (back) | `O(1)` amortized |
| `add(i, e)` (front, middle) | `O(N)` |
| `remove(size - 1)` (back) | `O(1)` |
| `remove(i)` (front, middle) | `O(N)` |
| `size()`, `isEmpty()` | `O(1)` |
| `contains`, `indexOf`, `remove(Object)` | `O(N)` |
| `equals`, `toArray`, `clear` | `O(N)` |
| `sort` | `O(N log N)` |

# Iterators

## The for-each loop

Works on arrays and on anything that implements `Iterable<E>`.

In [ ]:
import java.util.Iterator;

List<String> animals = new ArrayList<>(List.of("cat", "dog", "owl"));

for (String s : animals)
    System.out.println(s);

// what the compiler turns the loop into
Iterator<String> it = animals.iterator();
while (it.hasNext()) {
    String s = it.next();
    System.out.println(s);
}

## `Iterator<E>` and `Iterable<E>`

- `Iterator<E>`: `hasNext()`, `next()`. One pass over the collection; remembers its position.
- `next()` with nothing left: `NoSuchElementException`.
- `Iterable<E>`: a single method, `iterator()`, returning a **new** iterator.

## An iterator for `MyArrayListDouble`

In [ ]:
import java.util.NoSuchElementException;

class MyArrayListDoubleIterator implements Iterator<Double> {
    private final MyArrayListDouble list;
    private int next = 0;                   // position of the next element to return

    MyArrayListDoubleIterator(MyArrayListDouble list) { this.list = list; }

    public boolean hasNext() { return next < list.size(); }

    public Double next() {
        if (!hasNext()) throw new NoSuchElementException();
        double e = list.get(next);
        next++;
        return e;
    }
}

Iterator<Double> nit = new MyArrayListDoubleIterator(nums);
while (nit.hasNext())
    System.out.print(nit.next() + " ");

For the for-each loop, the list itself has to be `Iterable`:

In [ ]:
class IterableArrayListDouble extends MyArrayListDouble implements Iterable<Double> {
    public Iterator<Double> iterator() { return new MyArrayListDoubleIterator(this); }
}

IterableArrayListDouble temps = new IterableArrayListDouble();
temps.add(21.5);
temps.add(19.0);
temps.add(23.25);
for (double t : temps)
    System.out.println(t);

## Why iterators?

- One loop for every collection, whatever the storage.
- `get(i)` in a loop vs. an iterator on an array list: cost of each?
- Same question for a linked list, where `get(i)` walks from one end. *Next week.*

## Exercises

### 1. What does this print?

In [ ]:
List<String> aL = new ArrayList<>(List.of("hello", "big", "pink", "cat"));
Iterator<String> itr = aL.iterator();
System.out.println(itr.next());
itr.next();
System.out.println(itr.next());
Iterator<String> itr1 = aL.iterator();
Iterator<String> itr2 = aL.iterator();
System.out.println(itr1.next());
System.out.println(itr2.next());
System.out.println(itr.next());

### 2. Removing while iterating
What happens? Try again removing `"c"` instead of `"b"`. Explain both.

In [ ]:
List<String> xs = new ArrayList<>(List.of("a", "b", "c", "d"));
for (String v : xs)
    if (v.equals("b")) xs.remove(v);
System.out.println(xs);

### 3. `ArrayList` capacity
With the default constructor, what is the capacity after 1, 10, 11, 16 appends?

In [ ]:
// test helpers
import java.util.NoSuchElementException;

void check(boolean ok, String what) {
    System.out.println((ok ? "ok    " : "FAIL  ") + what);
}

// listOf(1, 2, 3): a new list with those elements
MyArrayListDouble listOf(double... values) {
    MyArrayListDouble l = new MyArrayListDouble();
    for (double v : values) l.add(v);
    return l;
}

### 4. `equals`
Override `equals(Object o)` in `MyArrayListDouble`: true if `o` is also a `MyArrayListDouble` with the same size and the same elements in the same order.

In [ ]:
MyArrayListDouble a = listOf(1, 2, 3);
check(a.equals(listOf(1, 2, 3)), "same elements");
check(a.equals(a), "itself");
check(!a.equals(listOf(1, 2)), "different size");
check(!a.equals(listOf(3, 2, 1)), "different order");
check(!a.equals("hello"), "not a list");
check(!a.equals(null), "null");
MyArrayListDouble big = new MyArrayListDouble(100);
big.add(1); big.add(2); big.add(3);
check(a.equals(big), "different capacity, same elements");

### 5. Remove every occurrence
`void removeAll(double e)`: remove every element equal to `e`.
- With `remove(pos)` in a loop: cost?
- Write an `O(N)` version with a single pass.

In [ ]:
MyArrayListDouble t = listOf(1, 2, 1, 1, 3, 1);
t.removeAll(1);
check(t.toString().equals("[2.0, 3.0]"), "several occurrences: " + t);
t = listOf(1, 1, 1);
t.removeAll(1);
check(t.isEmpty(), "everything removed: " + t);
t = listOf(2, 3);
t.removeAll(1);
check(t.toString().equals("[2.0, 3.0]"), "nothing to remove: " + t);
t = new MyArrayListDouble();
t.removeAll(1);
check(t.isEmpty(), "empty list");

### 6. `addAll`
`void addAll(int pos, MyListDouble other)`: insert all of `other` at `pos`. Shift the existing elements only once.

In [ ]:
MyArrayListDouble t = listOf(1, 2, 3);
t.addAll(1, listOf(7, 8));
check(t.toString().equals("[1.0, 7.0, 8.0, 2.0, 3.0]"), "middle: " + t);
t = listOf(1, 2);
t.addAll(0, listOf(9));
check(t.toString().equals("[9.0, 1.0, 2.0]"), "front: " + t);
t = listOf(1, 2);
t.addAll(2, listOf(9));
check(t.toString().equals("[1.0, 2.0, 9.0]"), "end: " + t);
t = listOf(1, 2);
t.addAll(1, new MyArrayListDouble());
check(t.toString().equals("[1.0, 2.0]"), "empty other: " + t);
t = new MyArrayListDouble(2);
t.add(1);
t.addAll(1, listOf(2, 3, 4, 5, 6));
check(t.toString().equals("[1.0, 2.0, 3.0, 4.0, 5.0, 6.0]"), "needs more than 2x capacity: " + t);
try {
    listOf(1).addAll(5, listOf(2));
    check(false, "invalid pos should throw");
} catch (IndexOutOfBoundsException e) {
    check(true, "invalid pos throws");
}

### 7. `reverse`
Reverse the list in place: `O(N)` time, `O(1)` extra space.

In [ ]:
MyArrayListDouble t = listOf(1, 2, 3, 4);
t.reverse();
check(t.toString().equals("[4.0, 3.0, 2.0, 1.0]"), "even size: " + t);
t = listOf(1, 2, 3);
t.reverse();
check(t.toString().equals("[3.0, 2.0, 1.0]"), "odd size: " + t);
t = listOf(5);
t.reverse();
check(t.toString().equals("[5.0]"), "single element: " + t);
t = new MyArrayListDouble();
t.reverse();
check(t.isEmpty(), "empty list");

### 8. Searching a sorted list
Add to `MyArrayListDouble`:
- `boolean isSorted()`
- `int binarySearch(double key)`: index of `key` or `-1`, assuming the list is sorted
- `int bestSearch(double key)`: checks `isSorted()`, then uses `binarySearch` or `indexOf`

Cost of `bestSearch`? Better, worse, or the same as `indexOf`?

In [ ]:
MyArrayListDouble s = listOf(1, 3, 5, 7, 9);
check(s.isSorted(), "sorted");
check(listOf(1, 1, 2).isSorted(), "sorted, with duplicates");
check(!listOf(1, 3, 2).isSorted(), "not sorted");
check(new MyArrayListDouble().isSorted() && listOf(5).isSorted(), "empty and single element");
check(s.binarySearch(1) == 0 && s.binarySearch(5) == 2 && s.binarySearch(9) == 4, "binarySearch finds first, middle, last");
check(s.binarySearch(0) == -1 && s.binarySearch(4) == -1 && s.binarySearch(10) == -1, "binarySearch, missing");
check(new MyArrayListDouble().binarySearch(1) == -1, "binarySearch, empty list");
check(s.bestSearch(7) == 3, "bestSearch, sorted");
check(listOf(3, 1, 2).bestSearch(2) == 2, "bestSearch, not sorted");

### 9. Iterator `remove`
`Iterator` also has `void remove()`: removes the last element returned by `next()`. Implement it in `MyArrayListDoubleIterator`. What must happen to `next`?

Calling `remove()` before `next()`, or twice in a row: `IllegalStateException`.

In [ ]:
MyArrayListDouble t = listOf(1, 2, 3, 4, 6);
Iterator<Double> itr = new MyArrayListDoubleIterator(t);
while (itr.hasNext())
    if (itr.next() % 2 == 0) itr.remove();
check(t.toString().equals("[1.0, 3.0]"), "removes the even elements: " + t);

Iterator<Double> it2 = new MyArrayListDoubleIterator(listOf(1, 2));
try {
    it2.remove();
    check(false, "remove before next should throw");
} catch (IllegalStateException e) {
    check(true, "remove before next throws");
}
it2.next();
it2.remove();
try {
    it2.remove();
    check(false, "remove twice should throw");
} catch (IllegalStateException e) {
    check(true, "remove twice throws");
}
check(it2.hasNext() && it2.next() == 2, "continues after remove");